# Step 7 — Asset Vulnerability Mapping

**Objective:** Overlay HIFLD grid infrastructure (substations, transmission lines) with LOCA2 projected heat hazard and CAISO wildfire zones to produce a county-level asset risk score.

**Output:** `data/processed/asset_risk_scores.csv` and choropleth maps.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import rankdata

from config.settings import RAW, PROCESSED, ERCOT_FIPS, CAISO_FIPS, ASSET_RISK_WEIGHTS
from src.viz.maps import choropleth_county


## 7.1 Load HIFLD substations

Download from: https://catalog.data.gov/dataset/electric-substations  
Save shapefile to `data/raw/hifld/Electric_Substations.shp`.

In [ ]:
substation_path = RAW['hifld'] / 'Electric_Substations.shp'
if substation_path.exists():
    substations = gpd.read_file(substation_path)
    print(f'Substations loaded: {len(substations):,}')
    print(substations.head())
else:
    print('Substation file not found.')
    substations = None


## 7.2 Load HIFLD transmission lines

Download from: https://gis-fws.opendata.arcgis.com/datasets/fws::us-electric-power-transmission-lines

In [ ]:
txline_path = RAW['hifld'] / 'Electric_Power_Transmission_Lines.shp'
if txline_path.exists():
    txlines = gpd.read_file(txline_path)
    print(f'Transmission line segments: {len(txlines):,}')
else:
    print('Transmission line file not found.')
    txlines = None


## 7.3 Load county geometries and spatial join

In [ ]:
tiger_path = RAW['hifld'] / 'tl_2023_us_county' / 'tl_2023_us_county.shp'
study_counties = None
if tiger_path.exists():
    counties = gpd.read_file(tiger_path)
    counties['fips'] = counties['GEOID'].astype(str).str.zfill(5)
    study_counties = counties[counties['fips'].isin(set(ERCOT_FIPS + CAISO_FIPS))].to_crs('EPSG:4326')
    print(f'Study counties: {len(study_counties)}')

if substations is not None and study_counties is not None:
    subs_proj = substations.to_crs('EPSG:4326')
    subs_county = gpd.sjoin(subs_proj, study_counties[['fips', 'geometry']], how='left', predicate='within')
    substation_counts = subs_county.groupby('fips').size().rename('n_substations')
    print('Top 10 counties by substation count:')
    print(substation_counts.nlargest(10))


## 7.4 Compute composite asset risk score

Score = 0.40 * heat_exposure_rank + 0.30 * wildfire_exposure_rank + 0.30 * asset_density_rank
(wildfire_exposure_rank = 0 for ERCOT counties)

In [ ]:
risk_frames = []
for fips_list, path, region in [
    (ERCOT_FIPS, PROCESSED['loca2_ercot'], 'ERCOT'),
    (CAISO_FIPS, PROCESSED['loca2_caiso'], 'CAISO')
]:
    if not path.exists():
        print(f'LOCA2 not found for {region}')
        continue
    proj = pd.read_csv(path)
    mid = proj[(proj['scenario'] == 'ssp585') & (proj['period_label'] == 'mid')].copy()
    if 'WSDI_delta' in mid.columns:
        mid['heat_exposure_rank'] = rankdata(mid['WSDI_delta'].fillna(0)) / len(mid)
    else:
        mid['heat_exposure_rank'] = 0.0
    mid['region'] = region

    # Asset density rank: substations per county area
    if 'substation_counts' in dir() and study_counties is not None:
        county_area = study_counties.set_index('fips')['geometry'].to_crs('EPSG:5070').area / 1e6  # km2
        sub_density = substation_counts.reindex(mid['fips'].values).fillna(0)
        area_km2 = county_area.reindex(mid['fips'].values).fillna(1)
        density_vals = (sub_density.values / area_km2.values)
        mid['asset_density_rank'] = rankdata(density_vals) / len(mid)
    else:
        mid['asset_density_rank'] = 0.5  # neutral placeholder if substations unavailable

    # Wildfire exposure rank: CAISO only (CAL FIRE FHSZ)
    fhsz_path = RAW['hifld'] / 'cal_fire_fhsz'
    if region == 'CAISO' and fhsz_path.exists() and txlines is not None and study_counties is not None:
        fhsz = gpd.read_file(fhsz_path).to_crs('EPSG:4326')
        high_fhsz = fhsz[fhsz.get('HAZ_CLASS', fhsz.columns[0]).isin(['Very High', 'VH'])]
        caiso_counties = study_counties[study_counties['fips'].isin(fips_list)].to_crs('EPSG:4326')
        caiso_lines = txlines.to_crs('EPSG:4326').clip(caiso_counties.union_all())
        lines_fhsz = gpd.overlay(caiso_lines, high_fhsz[['geometry']], how='intersection')
        lines_fhsz['county_fips'] = gpd.sjoin(
            lines_fhsz[['geometry']], caiso_counties[['fips', 'geometry']],
            how='left', predicate='intersects'
        )['fips'].values
        wf_km = lines_fhsz.to_crs('EPSG:5070').geometry.length.groupby(
            lines_fhsz['county_fips']
        ).sum() / 1000
        mid['wildfire_exposure_rank'] = rankdata(
            mid['fips'].map(wf_km).fillna(0)
        ) / len(mid)
    else:
        mid['wildfire_exposure_rank'] = 0.0  # zero for ERCOT or if FHSZ missing

    # Composite score using ASSET_RISK_WEIGHTS
    w = ASSET_RISK_WEIGHTS
    mid['composite_risk_score'] = (
        w['heat_exposure_rank']    * mid['heat_exposure_rank'] +
        w['wildfire_exposure_rank'] * mid['wildfire_exposure_rank'] +
        w['asset_density_rank']    * mid['asset_density_rank']
    )

    risk_frames.append(
        mid[['fips', 'region', 'heat_exposure_rank',
             'wildfire_exposure_rank', 'asset_density_rank',
             'composite_risk_score']].copy()
    )

if risk_frames:
    risk_df = pd.concat(risk_frames, ignore_index=True)
    print(f'Risk scores computed for {len(risk_df)} counties')
    print(risk_df.sort_values('composite_risk_score', ascending=False).head(10))


## 7.5 Choropleth map and save

In [ ]:
if 'risk_df' in dir() and study_counties is not None:
    risk_gdf = study_counties.merge(risk_df, on='fips', how='left')
    risk_df.to_csv(PROCESSED['asset_risk'], index=False)
    print('Saved:', PROCESSED['asset_risk'])

    fig, ax = plt.subplots(figsize=(14, 9))
    choropleth_county(
        risk_gdf, 'heat_exposure_rank',
        title='Heat Exposure Rank — ERCOT + CAISO (SSP5-8.5, mid-century WSDI change)',
        legend_label='Percentile rank',
        ax=ax,
    )
    plt.savefig('../data/processed/asset_heat_exposure_map.png', dpi=150, bbox_inches='tight')
    plt.show()
